In [ ]:
!pip uninstall -y paddleocr paddlepaddle paddlepaddle-gpu
!pip install -q easyocr

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import json
from pathlib import Path
from PIL import Image
from tqdm import tqdm
import easyocr


# =========================
# CONFIG
# =========================

TEST_IMG_DIR = Path("/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/test")

OUTPUT_JSON = Path("/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/test_ocr_words.json")
STATE_JSON = Path("/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/test_ocr_words.json.ocr_state.json")

MAX_IMAGES = 10
IMAGE_EXTS = ["*.jpg", "*.jpeg", "*.png", "*.webp"]


# =========================
# LOAD IMAGE LIST
# =========================

image_paths = []

for ext in IMAGE_EXTS:
    image_paths.extend(TEST_IMG_DIR.glob(ext))

image_paths = sorted(image_paths)[:MAX_IMAGES]

print("Folder:", TEST_IMG_DIR)
print("Found images:", len(image_paths))

for p in image_paths:
    print(p.name)

if len(image_paths) == 0:
    raise FileNotFoundError(f"Không tìm thấy ảnh trong folder: {TEST_IMG_DIR}")


# =========================
# INIT OCR CPU
# =========================
# gpu=False: bắt buộc chạy CPU

reader = easyocr.Reader(
    ["en"],
    gpu=False
)


# =========================
# HELPER FUNCTIONS
# =========================

def normalize_bbox_pixel_to_1000(bbox, width, height):
    x1, y1, x2, y2 = bbox

    x1 = int(round(1000 * x1 / width))
    y1 = int(round(1000 * y1 / height))
    x2 = int(round(1000 * x2 / width))
    y2 = int(round(1000 * y2 / height))

    x1 = max(0, min(1000, x1))
    y1 = max(0, min(1000, y1))
    x2 = max(0, min(1000, x2))
    y2 = max(0, min(1000, y2))

    if x2 < x1:
        x1, x2 = x2, x1

    if y2 < y1:
        y1, y2 = y2, y1

    return [x1, y1, x2, y2]


def poly_to_bbox(poly):
    xs = [float(p[0]) for p in poly]
    ys = [float(p[1]) for p in poly]

    return [
        min(xs),
        min(ys),
        max(xs),
        max(ys)
    ]


# =========================
# RESET OUTPUT
# =========================
# Để chạy sạch lại từ đầu, xóa file cũ nếu có.

if OUTPUT_JSON.exists():
    OUTPUT_JSON.unlink()

if STATE_JSON.exists():
    STATE_JSON.unlink()

all_pages = []


# =========================
# RUN OCR CPU
# =========================

for img_idx, img_path in enumerate(tqdm(image_paths, desc="Running EasyOCR CPU")):
    with Image.open(img_path) as img:
        width, height = img.size

    # EasyOCR result:
    # [
    #   (bbox_4_points, text, confidence),
    #   ...
    # ]
    result = reader.readtext(
        str(img_path),
        detail=1,
        paragraph=False
    )

    words = []
    bboxes = []
    confidences = []
    pixel_bboxes = []

    for poly, text, score in result:
        text = str(text).strip()

        if text == "":
            continue

        pixel_bbox = poly_to_bbox(poly)
        norm_bbox = normalize_bbox_pixel_to_1000(pixel_bbox, width, height)

        words.append(text)
        bboxes.append(norm_bbox)
        confidences.append(float(score))
        pixel_bboxes.append([int(round(v)) for v in pixel_bbox])

    page_data = {
        "image_id": img_idx,
        "file_name": img_path.name,
        "image_path": str(img_path),
        "width": width,
        "height": height,
        "words": words,
        "bboxes": bboxes,
        "confidences": confidences,
        "pixel_bboxes": pixel_bboxes,
        "bbox_scale": "0-1000",
        "ocr_engine": "easyocr_cpu"
    }

    all_pages.append(page_data)

    with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
        json.dump(all_pages, f, ensure_ascii=False, indent=2)

    with open(STATE_JSON, "w", encoding="utf-8") as f:
        json.dump(
            {
                "output_json": str(OUTPUT_JSON),
                "processed_images": len(all_pages),
                "last_file": img_path.name,
                "ocr_engine": "easyocr_cpu"
            },
            f,
            ensure_ascii=False,
            indent=2
        )

print("Done.")
print("Saved OCR JSON:", OUTPUT_JSON)
print("Saved OCR state:", STATE_JSON)
print("Total pages:", len(all_pages))

In [ ]:
!pip install -q transformers accelerate datasets seqeval

In [ ]:
import json
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
from tqdm import tqdm


# =========================
# CONFIG
# =========================

BASE_DIR = Path("/content/drive/MyDrive/Doan/Dataset_Layoutlmv3")

OCR_JSON_PATH = BASE_DIR / "test_ocr_words.json"
VIS_DIR = BASE_DIR / "test_ocr_visualizations"

VIS_DIR.mkdir(parents=True, exist_ok=True)


# =========================
# LOAD OCR JSON
# =========================

with open(OCR_JSON_PATH, "r", encoding="utf-8") as f:
    ocr_pages = json.load(f)

print("Number of OCR pages:", len(ocr_pages))


# =========================
# HELPER FUNCTIONS
# =========================

def bbox_1000_to_pixel(bbox, width, height):
    """
    Convert bbox từ thang 0-1000 về pixel ảnh gốc.
    bbox: [x1, y1, x2, y2]
    """
    x1, y1, x2, y2 = bbox

    x1 = int(round(x1 * width / 1000))
    y1 = int(round(y1 * height / 1000))
    x2 = int(round(x2 * width / 1000))
    y2 = int(round(y2 * height / 1000))

    x1 = max(0, min(width, x1))
    y1 = max(0, min(height, y1))
    x2 = max(0, min(width, x2))
    y2 = max(0, min(height, y2))

    return [x1, y1, x2, y2]


def get_font(size=18):
    font_candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
    ]

    for font_path in font_candidates:
        if Path(font_path).exists():
            return ImageFont.truetype(font_path, size)

    return ImageFont.load_default()


def draw_ocr_page(page, output_dir, show_text=True, show_index=True, min_conf=0.0):
    """
    Vẽ OCR bbox lên ảnh.
    - show_text=True: hiện text OCR
    - show_index=True: hiện thứ tự OCR item
    - min_conf: chỉ vẽ OCR item có confidence >= min_conf
    """
    image_path = page.get("image_path", None)

    if image_path is None:
        raise ValueError(f"Page không có image_path: {page.get('file_name')}")

    image = Image.open(image_path).convert("RGB")
    draw = ImageDraw.Draw(image)

    width, height = image.size

    words = page.get("words", [])
    bboxes = page.get("bboxes", [])
    confidences = page.get("confidences", [1.0] * len(words))

    font = get_font(size=18)

    for i, (word, bbox, conf) in enumerate(zip(words, bboxes, confidences)):
        conf = float(conf)

        if conf < min_conf:
            continue

        pixel_bbox = bbox_1000_to_pixel(bbox, width, height)
        x1, y1, x2, y2 = pixel_bbox

        # Vẽ khung OCR
        draw.rectangle(pixel_bbox, outline="red", width=3)

        label_parts = []

        if show_index:
            label_parts.append(str(i))

        if show_text:
            text = str(word)
            if len(text) > 30:
                text = text[:30] + "..."
            label_parts.append(text)

        label_parts.append(f"{conf:.2f}")

        label_text = " | ".join(label_parts)

        # Vẽ nền label
        text_x = x1
        text_y = max(0, y1 - 24)

        text_bbox = draw.textbbox((text_x, text_y), label_text, font=font)

        draw.rectangle(text_bbox, fill="yellow")
        draw.text((text_x, text_y), label_text, fill="black", font=font)

    output_path = output_dir / f"{Path(page['file_name']).stem}_ocr_vis.jpg"
    image.save(output_path, quality=95)

    return output_path


# =========================
# DRAW ALL OCR PAGES
# =========================

vis_paths = []

for page in tqdm(ocr_pages, desc="Drawing OCR visualization"):
    out_path = draw_ocr_page(
        page=page,
        output_dir=VIS_DIR,
        show_text=True,
        show_index=True,
        min_conf=0.0
    )
    vis_paths.append(out_path)

print("Saved OCR visualizations to:", VIS_DIR)

for p in vis_paths:
    print(p)

In [ ]:
from IPython.display import display, HTML
from PIL import Image
from pathlib import Path

# Nếu đã có biến vis_paths từ cell visualize trước đó thì dùng luôn.
# Nếu chưa có, load lại từ folder output.
VIS_DIR = Path("/content/drive/MyDrive/Doan/Dataset_Layoutlmv3/test_ocr_visualizations")

vis_paths = sorted(VIS_DIR.glob("*.jpg"))[:10]

print("Number of visualization images:", len(vis_paths))

for i, img_path in enumerate(vis_paths, start=1):
    display(HTML(f"<h3>{i}. {img_path.name}</h3>"))
    display(Image.open(img_path))

In [ ]:
import json
import torch
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from tqdm import tqdm
from collections import Counter
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification


# =========================
# PATH CONFIG
# =========================

BASE_DIR = Path("/content/drive/MyDrive/Doan/Dataset_Layoutlmv3")

CHECKPOINT_DIR = BASE_DIR / "layoutlmv3_checkpoints" / "checkpoint-1000"

OCR_JSON_PATH = BASE_DIR / "test_ocr_words.json"

LABEL2ID_PATH = BASE_DIR / "label_mapping" / "label2id.json"
ID2LABEL_PATH = BASE_DIR / "label_mapping" / "id2label.json"

OUTPUT_JSON = BASE_DIR / "test_layoutlmv3_predictions_chunked.json"
OUTPUT_CSV = BASE_DIR / "test_layoutlmv3_predictions_chunked.csv"


# =========================
# DEVICE
# =========================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


# =========================
# LOAD LABEL MAPPING
# =========================

with open(LABEL2ID_PATH, "r", encoding="utf-8") as f:
    label2id = json.load(f)

with open(ID2LABEL_PATH, "r", encoding="utf-8") as f:
    id2label_raw = json.load(f)

id2label = {int(k): v for k, v in id2label_raw.items()}


# =========================
# LOAD PROCESSOR + MODEL
# =========================

processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    CHECKPOINT_DIR,
    id2label=id2label,
    label2id=label2id
)

model.to(device)
model.eval()

print("Loaded:", CHECKPOINT_DIR)


# =========================
# HELPER FUNCTIONS
# =========================

def clean_bbox(bbox):
    x1, y1, x2, y2 = bbox

    x1 = int(max(0, min(1000, x1)))
    y1 = int(max(0, min(1000, y1)))
    x2 = int(max(0, min(1000, x2)))
    y2 = int(max(0, min(1000, y2)))

    if x2 < x1:
        x1, x2 = x2, x1

    if y2 < y1:
        y1, y2 = y2, y1

    return [x1, y1, x2, y2]


def get_valid_image_path(page):
    image_path = Path(page.get("image_path", ""))

    if image_path.exists():
        return image_path

    fallback_path = BASE_DIR / "test" / page["file_name"]

    if fallback_path.exists():
        return fallback_path

    raise FileNotFoundError(f"Không tìm thấy ảnh: {page['file_name']}")


def predict_chunk(image, chunk_words, chunk_boxes):
    """
    Predict cho một chunk OCR items.
    Trả về labels/scores theo đúng số chunk_words.
    """
    encoding = processor(
        image,
        chunk_words,
        boxes=chunk_boxes,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    word_ids = encoding.word_ids(batch_index=0)

    encoding_on_device = {
        k: v.to(device)
        for k, v in encoding.items()
    }

    with torch.no_grad():
        outputs = model(**encoding_on_device)

    logits = outputs.logits[0]
    probs = torch.softmax(logits, dim=-1)

    pred_ids = torch.argmax(probs, dim=-1).detach().cpu().numpy()
    pred_scores_token = torch.max(probs, dim=-1).values.detach().cpu().numpy()

    word_pred_ids = {}
    word_pred_scores = {}

    for token_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue

        word_pred_ids.setdefault(word_id, []).append(int(pred_ids[token_idx]))
        word_pred_scores.setdefault(word_id, []).append(float(pred_scores_token[token_idx]))

    pred_labels = []
    pred_scores = []

    for word_idx in range(len(chunk_words)):
        if word_idx not in word_pred_ids:
            pred_labels.append("TRUNCATED")
            pred_scores.append(0.0)
            continue

        majority_id = Counter(word_pred_ids[word_idx]).most_common(1)[0][0]

        pred_labels.append(id2label[majority_id])
        pred_scores.append(float(np.mean(word_pred_scores[word_idx])))

    return pred_labels, pred_scores


def predict_page_chunked(page, chunk_size=60, overlap=0):
    """
    Chia OCR items thành nhiều chunk để tránh max_length=512.
    chunk_size=120 an toàn cho OCR line-level.
    Nếu OCR word-level rất nhỏ thì có thể dùng 180.
    """
    image_path = get_valid_image_path(page)
    image = Image.open(image_path).convert("RGB")

    words = [str(w) for w in page.get("words", [])]
    boxes = [clean_bbox(b) for b in page.get("bboxes", [])]

    if len(words) != len(boxes):
        raise ValueError(
            f"Lỗi length {page['file_name']}: words={len(words)}, boxes={len(boxes)}"
        )

    final_labels = [None] * len(words)
    final_scores = [0.0] * len(words)

    chunk_infos = []

    start = 0

    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk_words = words[start:end]
        chunk_boxes = boxes[start:end]

        chunk_labels, chunk_scores = predict_chunk(
            image=image,
            chunk_words=chunk_words,
            chunk_boxes=chunk_boxes
        )

        for local_idx, global_idx in enumerate(range(start, end)):
            final_labels[global_idx] = chunk_labels[local_idx]
            final_scores[global_idx] = chunk_scores[local_idx]

        chunk_infos.append({
            "start": start,
            "end": end,
            "num_items": end - start
        })

        if overlap > 0:
            start = end - overlap
        else:
            start = end

    final_labels = [
        label if label is not None else "MISSING"
        for label in final_labels
    ]

    return {
        "file_name": page["file_name"],
        "image_path": str(image_path),
        "width": page["width"],
        "height": page["height"],
        "words": words,
        "bboxes": boxes,
        "pred_labels": final_labels,
        "pred_scores": final_scores,
        "num_words": len(words),
        "num_predicted_words": sum(1 for x in final_labels if x not in ["TRUNCATED", "MISSING"]),
        "chunk_size": chunk_size,
        "chunks": chunk_infos
    }


# =========================
# RUN CHUNKED PREDICTION
# =========================

with open(OCR_JSON_PATH, "r", encoding="utf-8") as f:
    ocr_pages = json.load(f)

ocr_pages = ocr_pages[:10]

all_predictions = []

for page in tqdm(ocr_pages, desc="Predicting LayoutLMv3 chunked"):
    pred_page = predict_page_chunked(
        page,
        chunk_size=60,
        overlap=0
    )
    all_predictions.append(pred_page)


# =========================
# SAVE JSON
# =========================

with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(all_predictions, f, ensure_ascii=False, indent=2)

print("Saved JSON:", OUTPUT_JSON)


# =========================
# SAVE CSV
# =========================

rows = []

for page in all_predictions:
    for i, word in enumerate(page["words"]):
        rows.append({
            "file_name": page["file_name"],
            "word_idx": i,
            "word": word,
            "bbox": page["bboxes"][i],
            "pred_label": page["pred_labels"][i],
            "pred_score": page["pred_scores"][i],
            "chunk_size": page["chunk_size"]
        })

df = pd.DataFrame(rows)
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

print("Saved CSV:", OUTPUT_CSV)
print("Total OCR items:", len(df))

display(df["pred_label"].value_counts().reset_index())
display(df.head(30))

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, HTML


# =========================
# CONFIG
# =========================

VIS_DIR = BASE_DIR / "test_layoutlmv3_visualizations_chunked"
VIS_DIR.mkdir(parents=True, exist_ok=True)


LABEL_COLORS = {
    "O": (128, 128, 128),
    "chart": (255, 165, 0),
    "figure": (128, 0, 128),
    "footer": (165, 42, 42),
    "header": (0, 102, 255),
    "ignore": (180, 180, 180),
    "table": (255, 0, 0),
    "table_text": (255, 0, 255),
    "text": (0, 170, 0),
    "toc": (0, 200, 200),
    "TRUNCATED": (0, 0, 0),
    "MISSING": (0, 0, 0)
}


def get_font(size=16):
    font_candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
    ]

    for font_path in font_candidates:
        if Path(font_path).exists():
            return ImageFont.truetype(font_path, size)

    return ImageFont.load_default()


def bbox_1000_to_pixel(bbox, width, height):
    x1, y1, x2, y2 = bbox

    x1 = int(round(x1 * width / 1000))
    y1 = int(round(y1 * height / 1000))
    x2 = int(round(x2 * width / 1000))
    y2 = int(round(y2 * height / 1000))

    x1 = max(0, min(width, x1))
    y1 = max(0, min(height, y1))
    x2 = max(0, min(width, x2))
    y2 = max(0, min(height, y2))

    return [x1, y1, x2, y2]


def draw_legend(draw, labels, font):
    labels = [x for x in labels if x in LABEL_COLORS]

    x = 20
    y = 20
    box_size = 18
    line_h = 25

    bg_w = 260
    bg_h = 20 + len(labels) * line_h + 20

    draw.rectangle([10, 10, 10 + bg_w, 10 + bg_h], fill=(255, 255, 255), outline=(0, 0, 0))

    for label in labels:
        color = LABEL_COLORS[label]
        draw.rectangle([x, y, x + box_size, y + box_size], fill=color, outline=(0, 0, 0))
        draw.text((x + 28, y - 2), label, fill=(0, 0, 0), font=font)
        y += line_h


def draw_layout_prediction(page_pred, output_dir, show_text=False, min_score=0.0):
    image = Image.open(page_pred["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(image)

    width, height = image.size

    font = get_font(size=16)
    small_font = get_font(size=14)

    used_labels = sorted(set(page_pred["pred_labels"]))

    for word, bbox, label, score in zip(
        page_pred["words"],
        page_pred["bboxes"],
        page_pred["pred_labels"],
        page_pred["pred_scores"]
    ):
        if float(score) < min_score:
            continue

        color = LABEL_COLORS.get(label, (255, 255, 0))
        pixel_bbox = bbox_1000_to_pixel(bbox, width, height)

        x1, y1, x2, y2 = pixel_bbox

        draw.rectangle(pixel_bbox, outline=color, width=3)

        label_text = f"{label}"

        if show_text:
            word_text = str(word)
            if len(word_text) > 22:
                word_text = word_text[:22] + "..."
            label_text = f"{label} | {word_text}"

        text_y = max(0, y1 - 20)
        text_bbox = draw.textbbox((x1, text_y), label_text, font=small_font)

        draw.rectangle(text_bbox, fill=color)
        draw.text((x1, text_y), label_text, fill=(0, 0, 0), font=small_font)

    draw_legend(draw, used_labels, font)

    output_path = output_dir / f"{Path(page_pred['file_name']).stem}_layoutlmv3_chunked_vis.jpg"
    image.save(output_path, quality=95)

    return output_path


# =========================
# DRAW + DISPLAY ALL 10
# =========================

layout_vis_paths = []

for page_pred in tqdm(all_predictions, desc="Drawing chunked LayoutLMv3 visualization"):
    out_path = draw_layout_prediction(
        page_pred=page_pred,
        output_dir=VIS_DIR,
        show_text=False,
        min_score=0.0
    )
    layout_vis_paths.append(out_path)

print("Saved to:", VIS_DIR)

for i, img_path in enumerate(layout_vis_paths, start=1):
    img = Image.open(img_path)

    max_width = 1100
    w, h = img.size

    if w > max_width:
        img = img.resize((max_width, int(h * max_width / w)))

    display(HTML(f"<h3>{i}. {Path(img_path).name}</h3>"))
    display(img)

In [ ]:
from PIL import Image, ImageDraw, ImageFont
from pathlib import Path
from tqdm import tqdm
from IPython.display import display, HTML


# =========================
# CONFIG
# =========================

CLEAR_VIS_DIR = BASE_DIR / "test_layoutlmv3_visualizations_clear"
CLEAR_VIS_DIR.mkdir(parents=True, exist_ok=True)

LABEL_COLORS = {
    "O": (128, 128, 128),
    "chart": (255, 165, 0),
    "figure": (128, 0, 128),
    "footer": (165, 42, 42),
    "header": (0, 102, 255),
    "ignore": (180, 180, 180),
    "table": (255, 0, 0),
    "table_text": (255, 0, 255),
    "text": (0, 180, 0),
    "toc": (0, 200, 200),
    "TRUNCATED": (0, 0, 0),
    "MISSING": (0, 0, 0)
}


def get_font(size=28):
    font_candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf"
    ]

    for font_path in font_candidates:
        if Path(font_path).exists():
            return ImageFont.truetype(font_path, size)

    return ImageFont.load_default()


def bbox_1000_to_pixel(bbox, width, height):
    x1, y1, x2, y2 = bbox

    x1 = int(round(x1 * width / 1000))
    y1 = int(round(y1 * height / 1000))
    x2 = int(round(x2 * width / 1000))
    y2 = int(round(y2 * height / 1000))

    x1 = max(0, min(width, x1))
    y1 = max(0, min(height, y1))
    x2 = max(0, min(width, x2))
    y2 = max(0, min(height, y2))

    return [x1, y1, x2, y2]


def draw_legend(draw, labels, font):
    labels = [x for x in labels if x in LABEL_COLORS]

    x = 30
    y = 30
    box_size = 28
    line_h = 40

    bg_w = 330
    bg_h = 30 + len(labels) * line_h + 30

    draw.rectangle(
        [15, 15, 15 + bg_w, 15 + bg_h],
        fill=(255, 255, 255),
        outline=(0, 0, 0),
        width=3
    )

    for label in labels:
        color = LABEL_COLORS[label]
        draw.rectangle(
            [x, y, x + box_size, y + box_size],
            fill=color,
            outline=(0, 0, 0),
            width=2
        )
        draw.text((x + 45, y - 5), label, fill=(0, 0, 0), font=font)
        y += line_h


def draw_layout_prediction_clear(page_pred, output_dir, show_text=False, min_score=0.0):
    image = Image.open(page_pred["image_path"]).convert("RGB")
    draw = ImageDraw.Draw(image)

    width, height = image.size

    font = get_font(size=28)
    small_font = get_font(size=22)

    used_labels = sorted(set(page_pred["pred_labels"]))

    for word, bbox, label, score in zip(
        page_pred["words"],
        page_pred["bboxes"],
        page_pred["pred_labels"],
        page_pred["pred_scores"]
    ):
        if float(score) < min_score:
            continue

        if label in ["TRUNCATED", "MISSING"]:
            continue

        color = LABEL_COLORS.get(label, (255, 255, 0))

        pixel_bbox = bbox_1000_to_pixel(bbox, width, height)
        x1, y1, x2, y2 = pixel_bbox

        # Bbox dày hơn để nhìn rõ
        draw.rectangle(pixel_bbox, outline=color, width=6)

        # Không hiện text OCR mặc định để ảnh không bị rối
        label_text = f"{label}"

        if show_text:
            word_text = str(word)
            if len(word_text) > 20:
                word_text = word_text[:20] + "..."
            label_text = f"{label} | {word_text}"

        text_x = x1
        text_y = max(0, y1 - 30)

        text_bbox = draw.textbbox((text_x, text_y), label_text, font=small_font)

        # Nền label rõ hơn
        draw.rectangle(text_bbox, fill=color)
        draw.text((text_x, text_y), label_text, fill=(0, 0, 0), font=small_font)

    draw_legend(draw, used_labels, font)

    # Lưu PNG để không bị nén mờ như JPG
    output_path = output_dir / f"{Path(page_pred['file_name']).stem}_layoutlmv3_clear.png"
    image.save(output_path)

    return output_path


# =========================
# DRAW ALL 10 IMAGES
# =========================

clear_vis_paths = []

for page_pred in tqdm(all_predictions, desc="Drawing clear LayoutLMv3 visualization"):
    out_path = draw_layout_prediction_clear(
        page_pred=page_pred,
        output_dir=CLEAR_VIS_DIR,
        show_text=False,
        min_score=0.0
    )
    clear_vis_paths.append(out_path)

print("Saved clear visualizations to:", CLEAR_VIS_DIR)

for p in clear_vis_paths:
    print(p)

In [ ]:
from IPython.display import display, HTML
from PIL import Image
from pathlib import Path

for i, img_path in enumerate(clear_vis_paths, start=1):
    img = Image.open(img_path)

    # Tăng max_width lên để đỡ mờ
    max_width = 1800
    w, h = img.size

    if w > max_width:
        new_h = int(h * max_width / w)
        img = img.resize((max_width, new_h), resample=Image.Resampling.LANCZOS)

    display(HTML(f"<h3>{i}. {Path(img_path).name}</h3>"))
    display(img)